
 ### <font color="orange">🧠 GIAI ĐOẠN 3: KỶ NGUYÊN LLMs</font>
 #### Bài 27: Lắp ráp Transformer Block (Cỗ máy tạo ra GPT & BERT)

 Ngày hôm qua em đã chế tạo thành công "Lò phản ứng" Attention. Nhưng lò phản ứng cắm điện trần sẽ rất dễ nổ (Lỗi Gradient).
 Google đã bọc nó vào một cấu trúc cực kỳ an toàn gọi là **Transformer Block**.

 **1. Cấu trúc của một Transformer Block:**
 Dữ liệu đi vào sẽ đi qua 2 trạm kiểm soát:
 - **Trạm 1:** Multi-Head Attention (Để các từ giao tiếp và chú ý lẫn nhau).
 - **Trạm 2:** Feed Forward Neural Network (Một mạng nơ-ron nhỏ để mỗi từ tự suy ngẫm và chắt lọc thông tin riêng của nó).

 **2. Phép màu Residual Connection (Kết nối tắt/Phần dư):**
 Khi đi qua mỗi trạm (ví dụ qua trạm Attention), nhỡ đâu mạng nơ-ron học "ngu đi" thì sao?
 Khắc phục: Người ta sẽ bắc một cái cầu "vượt tuyến", lấy dữ liệu GỐC cộng thẳng vào kết quả SAU KHI qua trạm.
 > $ \color{orange} Output = LayerNorm(Trạm(x) + x) $
 
 Chính cơ chế cộng đơn giản này (`+ x`) là chìa khóa thần kỳ giúp ChatGPT có thể xếp chồng 96 lớp (Layers) lên nhau mà mạng không bị sập hay mất trí nhớ!

 **3. Sự phân chia ranh giới: BERT và GPT**
 Từ kiến trúc Transformer Block này, giới AI chia làm 2 phe:
 - **Phe BERT (Dùng Encoder):** Đọc cả câu cùng lúc, nhìn trái nhìn phải. Cực giỏi việc Đọc hiểu văn bản, Phân loại, Tìm kiếm (Google Search đang dùng BERT).
 - **Phe GPT (Dùng Decoder):** Bị "bịt mắt" (Masked), chỉ được phép nhìn các từ trong quá khứ để đoán từ tiếp theo. Cực giỏi Sinh chữ, Chatbot.



In [1]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super(TransformerBlock, self).__init__()
        # Trạm 1: Attention đa luồng (Sư phụ dùng đồ build sẵn của PyTorch cho nhanh)
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        
        # Trạm 2: Mạng nơ-ron truyền thẳng (Feed Forward)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # ====================================================================
        # NHIỆM VỤ CỦA EM: LẮP RÁP CÁC TRẠM VÀ THIẾT LẬP KẾT NỐI TẮT (RESIDUAL)
        # Gợi ý: Hãy nhìn kỹ công thức màu cam ở trên phần lý thuyết!
        # ====================================================================
        
        # --- TRẠM 1: ATTENTION ---
        # 1. Đưa x vào Attention (Vì x tự chú ý đến chính nó nên Q=x, K=x, V=x)
        attn_output, _ = self.attention(x, x, x)
        
        # 2. Lắp Cầu vượt (Residual) và Chuẩn hóa
        # TỰ CODE VÀO ĐÂY: Lấy kết quả attn_output CỘNG với đầu vào gốc x, 
        # sau đó bọc tất cả lại và đưa qua hàm chuẩn hóa self.norm1(...)
        out_1 = self.norm1(attn_output + x)
        
        # --- TRẠM 2: FEED FORWARD ---
        # 3. Đưa out_1 qua mạng Feed Forward
        ff_output = self.feed_forward(out_1)
        
        # 4. Lắp Cầu vượt (Residual) lần 2
        # TỰ CODE VÀO ĐÂY: Lấy ff_output CỘNG với đầu vào của trạm 2 (chính là out_1),
        # sau đó bọc lại và đưa qua self.norm2(...)
        out_2 = self.norm1(ff_output + out_1)
        
        return out_2

# %%
# ====================================================================
# KHU VỰC KIỂM THỬ (UNIT TESTS)
# ====================================================================
def main():
    print("--- THỬ THÁCH ĐÓNG GÓI TRANSFORMER BLOCK ---")
    
    # Giả lập 1 câu văn có 5 từ, mỗi từ là vector 16 chiều
    batch_size = 1
    seq_len = 5
    d_model = 16
    num_heads = 4
    
    # Tạo tensor mô phỏng câu văn
    x = torch.rand(batch_size, seq_len, d_model)
    print(f"Kích thước đầu vào (Input): {x.shape} -> (Batch, Từ, Chiều vector)")
    
    # Đúc ra một khối Transformer
    block = TransformerBlock(d_model, num_heads)
    
    try:
        output = block(x)
        print(f"Kích thước đầu ra (Output): {output.shape}")
        
        if output.shape == x.shape:
            print("\n✅ KIỂM ĐỊNH THÀNH CÔNG: KHỚP KÍCH THƯỚC!")
            print("Chúc mừng! Em đã biết cách 'đóng gói' Attention vào trong Transformer Block.")
            print("💡 Bí mật: Nếu em dùng vòng lặp for gọi `block` này 96 lần liên tiếp, em vừa tạo ra kiến trúc cốt lõi của GPT-3 đấy!")
        else:
            print("❌ Lỗi kích thước. Đầu vào và đầu ra của Block phải giống hệt nhau về Shape.")
    except Exception as e:
        print(f"❌ Lỗi code: {e}")

if __name__ == "__main__":
    main()

--- THỬ THÁCH ĐÓNG GÓI TRANSFORMER BLOCK ---
Kích thước đầu vào (Input): torch.Size([1, 5, 16]) -> (Batch, Từ, Chiều vector)
Kích thước đầu ra (Output): torch.Size([1, 5, 16])

✅ KIỂM ĐỊNH THÀNH CÔNG: KHỚP KÍCH THƯỚC!
Chúc mừng! Em đã biết cách 'đóng gói' Attention vào trong Transformer Block.
💡 Bí mật: Nếu em dùng vòng lặp for gọi `block` này 96 lần liên tiếp, em vừa tạo ra kiến trúc cốt lõi của GPT-3 đấy!
